# ML-04 — Search Intelligence Data Contract

This notebook completes the assignment: describe the lane, verify three facts with queries (mid-panel month=2026-03), build five features and demonstrate a deliberate leakage trap.

> I read `skills/README.md` and will inspect the `FlyRank/internship-warehouse` dataset on Hugging Face programmatically.

## 1) Unit of analysis + time window (plain words)

1) One row means: a single (client_hash_id, content_hash_id, report_date) observation — i.e., one client’s content aggregate on a single calendar date.
2) Table(s) used: `fact_content_daily_performance/month=2026-03/data_0.parquet` from the `FlyRank/internship-warehouse` dataset (discovered programmatically).
3) Time window: month = 2026-03 for exploration and feature building (the notebook loads and filters that month).
4) Predict/rank (label): whether the row had any GSC clicks (derived as `gsc_clicks>0`).
5) Deliberately excluded: raw client identifiers and any cross-day future aggregates that would leak outcomes or PII.

In [1]:
# Discovery: list files in the Hugging Face dataset and pick candidates mentioning 2026-03
import os
from huggingface_hub import HfApi

def load_token(env_path='.env'):
    if not os.path.exists(env_path):
        return None
    with open(env_path, 'r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('=', 1)
            if len(parts)!=2:
                continue
            k, v = parts[0].strip(), parts[1].strip().strip('"')
            if k in ('HF_TOKEN','HUGGINGFACE_HUB_TOKEN','HF_HUB_TOKEN','HUGGINGFACE_TOKEN'):
                return v
    return None

token = load_token()
api = HfApi()
repo_id = 'FlyRank/internship-warehouse'
print('Listing files in dataset:', repo_id)
files = api.list_repo_files(repo_id, repo_type='dataset', token=token)
month = '2026-03'
candidates = [f for f in files if month in f or '2026-03' in f]
print('Found', len(candidates), 'files mentioning', month)
# show some examples
for f in candidates[:40]:
    print('-', f)
# fallback to top-level listing if empty
if not candidates:
    print('No month-specific files found; showing top-level files (sample):')
    for f in files[:60]:
        print('-', f)

CANDIDATE_FILES = candidates or files
# keep CANDIDATE_FILES available for subsequent cells
CANDIDATE_FILES[:8]

c:\Users\HP\OneDrive - Higher Education Commission\Desktop\summer2026\Flyrank\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Listing files in dataset: FlyRank/internship-warehouse
Found 1 files mentioning 2026-03
- fact_content_daily_performance/month=2026-03/data_0.parquet


['fact_content_daily_performance/month=2026-03/data_0.parquet']

## 2) Fields: feature / label / context / excluded (plain words)

- Label: `gsc_clicks>0` (binary) — derived from `gsc_clicks` when explicit label is not present.
- Features (five chosen): `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (or `gsc_sum_position` / impressions), `ga4_pageviews`, `client_has_gsc` (availability flag).
- Context: `report_date`, `client_hash_id`, `content_hash_id`, `country`, `device` (when present).
- Excluded: any raw client identifiers beyond the hashed `client_hash_id`, and any `future_` columns that would reveal post-decision outcomes.

## 3) Verify three facts with queries (on month=2026-03) — code below runs these checks:
1. Grain: confirm one row corresponds to (date, query, page) by checking uniqueness.
2. Row count + date span: how many rows in 2026-03 and the min/max date.
3. Availability: filter boolean-like columns with `IS TRUE` and report counts.

In [2]:
# Verification queries: download a candidate tabular file, read it and run the three checks
import os
import pandas as pd
from huggingface_hub import hf_hub_download

# ensure token is loaded (try fallback to repo .env absolute path)
if token is None:
    alt_env = r"c:\Users\HP\OneDrive - Higher Education Commission\Desktop\summer2026\Flyrank\Flyrank_ML_Internship_Hamza\.env"
    if os.path.exists(alt_env):
        with open(alt_env, 'r', encoding='utf-8') as f:
            for line in f:
                if '=' in line:
                    k, v = line.split('=', 1)
                    k = k.strip()
                    v = v.strip().strip('"').strip('\'')
                    if k in ('HF_TOKEN','HUGGINGFACE_HUB_TOKEN','HF_HUB_TOKEN','HUGGINGFACE_TOKEN'):
                        token = v
                        break
    if token:
        os.environ['HUGGINGFACE_HUB_TOKEN'] = token

# choose a candidate file that looks tabular
chosen = None
for f in CANDIDATE_FILES:
    if ('2026-03' in f) and (f.endswith('.parquet') or f.endswith('.csv') or f.endswith('.tsv')):
        chosen = f
        break
if not chosen:
    for f in CANDIDATE_FILES:
        if f.endswith('.parquet') or f.endswith('.csv') or f.endswith('.tsv'):
            chosen = f
            break
if not chosen:
    raise SystemExit('No tabular candidate file found; please run the discovery cell first')

print('Chosen file:', chosen)
local_path = hf_hub_download(repo_id='FlyRank/internship-warehouse', filename=chosen, repo_type='dataset', token=token)
print('Downloaded to:', local_path)
# read
if chosen.endswith('.parquet'):
    df = pd.read_parquet(local_path)
else:
    df = pd.read_csv(local_path)

print('Columns sample:', list(df.columns)[:40])
print('Total rows in file:', len(df))

# 1) Grain check: find likely date/query/page columns
possible_date = [c for c in df.columns if 'date' in c.lower() or c.lower()=='day']
possible_query = [c for c in df.columns if 'query' in c.lower()]
possible_page = [c for c in df.columns if 'page' in c.lower() or 'path' in c.lower()]
print('possible_date=', possible_date, 'possible_query=', possible_query, 'possible_page=', possible_page)
date_col = possible_date[0] if possible_date else None
query_col = possible_query[0] if possible_query else None
page_col = possible_page[0] if possible_page else None
if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    print('date range (file):', df[date_col].min(), '->', df[date_col].max())

if date_col and query_col and page_col:
    total = len(df)
    unique_rows = df[[date_col, query_col, page_col]].drop_duplicates().shape[0]
    print('Total rows:', total, 'Unique (date,query,page):', unique_rows)
    if unique_rows==total:
        print('Grain check PASSED: one row == (date,query,page)')
    else:
        print('Grain check WARNING: duplicates exist at (date,query,page)')
else:
    print('Could not perform exact grain uniqueness check: missing one of date/query/page')

# 2) Row count + date span for 2026-03
if date_col is not None:
    df_month = df[df[date_col].dt.to_period('M')==pd.Period('2026-03')]
    print('Rows in 2026-03:', len(df_month))
    if len(df_month):
        print('2026-03 span:', df_month[date_col].min(), '->', df_month[date_col].max())
else:
    print('No date column — cannot filter to 2026-03')

# 3) Availability: boolean-like columns and how many rows survive IS TRUE
bool_cols = []
for c in df.columns:
    col = df[c].dropna()
    if col.empty:
        continue
    try:
        uniq = set(col.unique())
    except Exception:
        uniq = set()
    if len(uniq)<=3 and uniq.issubset({0,1,True,False,'0','1'}):
        bool_cols.append(c)
bool_cols += [c for c in df.columns if c.lower().startswith('is_') or 'available' in c.lower()]
bool_cols = list(dict.fromkeys(bool_cols))
print('Boolean-like columns:', bool_cols)
for c in bool_cols:
    try:
        survivors = int((df[c]==True).sum())
        print(f"{c} IS TRUE -> {survivors} rows (of {len(df)})")
    except Exception:
        print('Could not compute IS TRUE for', c)

# keep DF and DF_MONTH available for the feature-building cell
DF = df
DF_MONTH = df_month if 'df_month' in locals() else df
DF_MONTH.head().astype(str).iloc[:8,:8]

Chosen file: fact_content_daily_performance/month=2026-03/data_0.parquet
Downloaded to: C:\Users\HP\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet
Columns sample: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Total rows in file: 9841378
possible_date= ['report_date'] possible_query= [] possible_page= ['ga4_pageviews']
date range (file): 2026-03-01 00:00:00 -> 2026-03-31 00:00:00
Could not p

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,NaN,20
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,NaN,1
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,NaN,125
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,NaN,7
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,NaN,11


## 4) Five features (build a small feature frame) and the trap

Below I build five features from the mid-panel month (2026-03). For each feature I list when it is knowable at decision time. Then I intentionally add a label-derived leakage column, show the (inflated) quick check, and remove it to keep the honest number.

In [ ]:
# Feature construction (five features) and deliberate leakage experiment
import numpy as np
from sklearn.metrics import roc_auc_score

df = DF_MONTH.copy()
# helper to find candidate cols
def pick(cols, candidates):
    for pat in candidates:
        for c in cols:
            if pat in c.lower():
                return c
    return None

cols = list(df.columns)
imp_col = pick(cols, ['gsc_impressions','impression','impr'])
click_col = pick(cols, ['gsc_clicks','click'])
pos_col = pick(cols, ['gsc_avg_position','gsc_sum_position','position','pos','rank'])
pageviews_col = pick(cols, ['ga4_pageviews','page_views','pageviews','page_view','page_depth'])

# Build features with safe fallbacks
F = pd.DataFrame()
F['gsc_impressions'] = df[imp_col].fillna(0) if imp_col in df.columns else 0
F['gsc_clicks'] = df[click_col].fillna(0) if click_col in df.columns else 0
F['ctr'] = (F['gsc_clicks'] / F['gsc_impressions']).replace([np.inf, -np.inf], 0).fillna(0)
if pos_col in df.columns:
    F['position'] = df[pos_col].fillna(99)
else:
    F['position'] = 99
F['ga4_pageviews'] = df[pageviews_col].fillna(0) if pageviews_col in df.columns else 0

# Attach context columns so we can inspect grain keys later
for k in ['report_date','client_hash_id','content_hash_id']:
    if k in df.columns and k not in F.columns:
        F[k] = df[k]

print('Feature frame shape:', F.shape)
display(F.head().astype(str).iloc[:8,:8])

# Feature availability lines (one-liners):
avail = []
avail.append(('gsc_impressions', 'available when GSC daily impressions are present in the row'))
avail.append(('gsc_clicks', 'available when GSC click counters are emitted in the daily row'))
avail.append(('ctr', 'computable from impressions+clicks at decision time'))
avail.append(('position', 'available when GSC provides average/position sums for the row'))
avail.append(('ga4_pageviews', 'available when GA4 page-level metrics are present for the same date'))
for name, text in avail:
    print(f"- {name}: {text}")

# Deliberate leakage trap: if a label exists, create a leak column identical to label, show inflated metric, then remove it
label_candidates = [c for c in df.columns if 'label' in c.lower() or 'is_click' in c.lower() or ('gsc_clicks'==c.lower())]
label_col = None
for c in label_candidates:
    if c in df.columns:
        if 'is_' in c.lower() or 'label' in c.lower() or c.lower()=='is_click':
            label_col = c
            break
        label_col = c

if label_col is None and 'gsc_clicks' in df.columns:
    df['_label_derived'] = (df['gsc_clicks']>0).astype(int)
    label_col = '_label_derived'

if label_col is not None:
    print('Using label column:', label_col)
    # use binary label for evaluation
    y = (df[label_col]>0).astype(int) if label_col!='_label_derived' else df[label_col].astype(int)
    F['leakage_feature'] = y.values
    try:
        auc_leak = roc_auc_score(y, F['leakage_feature'])
        print('AUC with leakage_feature present (inflated):', auc_leak)
    except Exception as e:
        print('Could not compute AUC:', e)
    if 'ctr' in F.columns:
        try:
            auc_no_leak = roc_auc_score(y, F['ctr'])
            print('AUC without leakage (ctr as signal):', auc_no_leak)
        except Exception as e:
            print('Could not compute baseline AUC:', e)
    F = F.drop(columns=[c for c in F.columns if 'leak' in c.lower()])
    print('Leakage column removed; keeping honest feature frame shape:', F.shape)
else:
    print('No label candidate found — skipped leakage experiment (safe).')

Feature frame shape: (9841378, 8)


,gsc_impressions,gsc_clicks,ctr,position,ga4_pageviews,report_date,client_hash_id,content_hash_id
0,20,0,0.0,3.35,0.0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,1,0,0.0,0.0,0.0,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067
2,125,1,0.008,4.928,0.0,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916
3,7,0,0.0,4.0,0.0,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e
4,11,0,0.0,2.272727272727273,0.0,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72


- gsc_impressions: available when GSC daily impressions are present in the row
- gsc_clicks: available when GSC click counters are emitted in the daily row
- ctr: computable from impressions+clicks at decision time
- position: available when GSC provides average/position sums for the row
- ga4_pageviews: available when GA4 page-level metrics are present for the same date
Using label column: gsc_clicks
Could not compute AUC: multi_class must be in ('ovo', 'ovr')
Could not compute baseline AUC: multi_class must be in ('ovo', 'ovr')
Leakage column removed; keeping honest feature frame shape: (9841378, 8)


## 5) Data limits and self-check

Limitation (named): this lane cannot reconstruct user sessions or identify individual users — warehouse rows are client-content-date aggregates and do not contain PII.

Self-check before submitting:

- [ ] Every section above contains a plain-words answer and the code cell that backs it.
- [ ] The notebook runs top-to-bottom with no errors (Runtime → Run all).
- [ ] No private tokens or secrets are printed in outputs.
- [ ] I committed this executed notebook under `work/notebooks/w03_data_contract.ipynb`.